UK Police Crime Data Engineering Project

Project Overview
This project will analyse recorded crime across Lincolnshire, Metropolitan, South Yorkshire and West Midlands Police forces. Population estimates and neighbourhood deprivation data is also used to enrich the data and its findings. 

Data imports and configuration

In [2]:
from pathlib import Path 
import pandas as pd 


In [10]:
DATA_DIR = Path("data/raw_data")

CRIME_DIR = DATA_DIR / "crime_data"
POPULARION_DIR = DATA_DIR / "population_data"
DEPRIVATION_DIR = DATA_DIR / "deprivation_data"

In [18]:
crime_files = sorted(CRIME_DIR.rglob("*.csv"))

print("Number of CSV files:", len(crime_files))

Number of CSV files: 144


In [19]:
# ensure that the files can be navigated between and view the format 
for file in crime_files[:10]:
    print(file)

data/raw_data/crime_data/2023-09/2023-09-lincolnshire-street.csv
data/raw_data/crime_data/2023-09/2023-09-metropolitan-street.csv
data/raw_data/crime_data/2023-09/2023-09-south-yorkshire-street.csv
data/raw_data/crime_data/2023-09/2023-09-west-midlands-street.csv
data/raw_data/crime_data/2023-10/2023-10-lincolnshire-street.csv
data/raw_data/crime_data/2023-10/2023-10-metropolitan-street.csv
data/raw_data/crime_data/2023-10/2023-10-south-yorkshire-street.csv
data/raw_data/crime_data/2023-10/2023-10-west-midlands-street.csv
data/raw_data/crime_data/2023-11/2023-11-lincolnshire-street.csv
data/raw_data/crime_data/2023-11/2023-11-metropolitan-street.csv


In [20]:
# verify the data and ensure that it covers the months and forces intended 
months = sorted({file.name[:7] for file in crime_files})

forces = sorted({
    file.name.removesuffix(".csv").split("-street")[0][8:]
    for file in crime_files
})

print("Number of months:", len(months))
print("First month:", months[0])
print("Last month:", months[-1])
print("Police forces:", forces)

Number of months: 36
First month: 2023-09
Last month: 2026-08
Police forces: ['lincolnshire', 'metropolitan', 'south-yorkshire', 'west-midlands']


In [21]:
sample_file = crime_files[0]
sample_df = pd.read_csv(sample_file)
print("File:", sample_file.name)
print("Shape:", sample_df.shape)
sample_df.head()

File: 2023-09-lincolnshire-street.csv
Shape: (5710, 12)


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,5e8004e8076b16766a773df834cb6e5f80db26a648d246...,2023-09,Lincolnshire Police,Lincolnshire Police,0.063610,52.984287,On or near Broughton'S Lane,E01026007,Boston 001A,Violence and sexual offences,Unable to prosecute suspect,NaN
1,3c907650b94dd38881aee7d919292f9eaabcbeaab0aa78...,2023-09,Lincolnshire Police,Lincolnshire Police,0.051572,52.967296,On or near Church End Road,E01026008,Boston 001B,Vehicle crime,Unable to prosecute suspect,NaN
2,NaN,2023-09,Lincolnshire Police,Lincolnshire Police,0.079021,52.960553,On or near Parking Area,E01026008,Boston 001B,Anti-social behaviour,NaN,NaN
3,NaN,2023-09,Lincolnshire Police,Lincolnshire Police,0.061304,52.981234,On or near St Andrew'S Road,E01026008,Boston 001B,Anti-social behaviour,NaN,NaN
4,NaN,2023-09,Lincolnshire Police,Lincolnshire Police,0.060653,52.948456,On or near Cropper'S Lane,E01026008,Boston 001B,Anti-social behaviour,NaN,NaN


In [34]:
sample_df.info()
sample_df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5710 entries, 0 to 5709
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Crime ID               4715 non-null   object 
 1   Month                  5710 non-null   object 
 2   Reported by            5710 non-null   object 
 3   Falls within           5710 non-null   object 
 4   Longitude              5710 non-null   float64
 5   Latitude               5710 non-null   float64
 6   Location               5710 non-null   object 
 7   LSOA code              5710 non-null   object 
 8   LSOA name              5710 non-null   object 
 9   Crime type             5710 non-null   object 
 10  Last outcome category  4715 non-null   object 
 11  Context                0 non-null      float64
dtypes: float64(3), object(9)
memory usage: 535.4+ KB


Crime ID                  995
Month                       0
Reported by                 0
Falls within                0
Longitude                   0
Latitude                    0
Location                    0
LSOA code                   0
LSOA name                   0
Crime type                  0
Last outcome category     995
Context                  5710
dtype: int64

In [26]:
sample_df[sample_df['Crime ID'].isna()]['Crime type'].value_counts()

Crime type
Anti-social behaviour    995
Name: count, dtype: int64

In [27]:
sample_df["Context"].nunique(dropna=False)

1

In [ ]:
# load all of the files into a data frame 
crime_dataframes = []

for file in crime_files:
    df = pd.read_csv(file)
    crime_dataframes.append(df)

crime_df = pd.concat(crime_dataframes, ignore_index=True)

print(len(crime_df))
print(len(crime_df.columns))
print(len(crime_dataframes))

5150729
12
144


In [31]:
missing_values = crime_df.isna().sum()
print(missing_values[missing_values > 0])


Crime ID                 892770
Longitude                 35735
Latitude                  35735
LSOA code                 35736
LSOA name                 35736
Last outcome category    892770
dtype: int64


In [ ]:
# drop the context column (all fields are NaN)
crime_df = crime_df.drop(columns= 'Context')
print(missing_values[missing_values > 0])

Crime ID                 892770
Longitude                 35735
Latitude                  35735
LSOA code                 35736
LSOA name                 35736
Last outcome category    892770
dtype: int64


In [ ]:
# find the rows where crime id is missing and find the crime type based on this
crime_df[crime_df["Crime ID"].isna()]["Crime type"].value_counts()

Crime type
Anti-social behaviour    892770
Name: count, dtype: int64

In [ ]:
# find exact duplicates 
print("Duplicate rows:", crime_df.duplicated().sum())

Duplicate rows: 370933


In [ ]:
# find all of the rows that are duplicated
crime_df[crime_df.duplicated(keep=False)].sort_values(
    by=["Month", "Crime type"]
).head(10)

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category
111,NaN,2023-09,Lincolnshire Police,Lincolnshire Police,-0.009965,52.991122,On or near Hospital,E01026021,Boston 003B,Anti-social behaviour,NaN
112,NaN,2023-09,Lincolnshire Police,Lincolnshire Police,-0.009965,52.991122,On or near Hospital,E01026021,Boston 003B,Anti-social behaviour,NaN
113,NaN,2023-09,Lincolnshire Police,Lincolnshire Police,-0.009965,52.991122,On or near Hospital,E01026021,Boston 003B,Anti-social behaviour,NaN
194,NaN,2023-09,Lincolnshire Police,Lincolnshire Police,-0.021274,52.981415,On or near Tawney Street,E01026036,Boston 003D,Anti-social behaviour,NaN
195,NaN,2023-09,Lincolnshire Police,Lincolnshire Police,-0.024807,52.978075,On or near Church Lane,E01026036,Boston 003D,Anti-social behaviour,NaN
196,NaN,2023-09,Lincolnshire Police,Lincolnshire Police,-0.021274,52.981415,On or near Tawney Street,E01026036,Boston 003D,Anti-social behaviour,NaN
197,NaN,2023-09,Lincolnshire Police,Lincolnshire Police,-0.024122,52.977426,On or near Fish Hill,E01026036,Boston 003D,Anti-social behaviour,NaN
198,NaN,2023-09,Lincolnshire Police,Lincolnshire Police,-0.024807,52.978075,On or near Church Lane,E01026036,Boston 003D,Anti-social behaviour,NaN
200,NaN,2023-09,Lincolnshire Police,Lincolnshire Police,-0.024122,52.977426,On or near Fish Hill,E01026036,Boston 003D,Anti-social behaviour,NaN
244,NaN,2023-09,Lincolnshire Police,Lincolnshire Police,-0.026126,52.977108,On or near Shopping Area,E01034681,Boston 003E,Anti-social behaviour,NaN


In [ ]:
# find the rows that are part of the duplicates
duplicate_check = crime_df[
    crime_df.duplicated(keep=False)
]

print("Rows involved in duplicates:", len(duplicate_check))

print(
    duplicate_check.groupby(["Month", "Reported by"])
    .size()
    .head(20)
)

Rows involved in duplicates: 544414
Month    Reported by                
2023-09  Lincolnshire Police              385
         Metropolitan Police Service    14370
         South Yorkshire Police           797
         West Midlands Police             979
2023-10  Lincolnshire Police              397
         Metropolitan Police Service    13582
         South Yorkshire Police           819
         West Midlands Police             867
2023-11  Lincolnshire Police              339
         Metropolitan Police Service    12553
         South Yorkshire Police           604
         West Midlands Police             681
2023-12  Lincolnshire Police              317
         Metropolitan Police Service    11518
         South Yorkshire Police           606
         West Midlands Police             576
2024-01  Lincolnshire Police              337
         Metropolitan Police Service    11717
         South Yorkshire Police           719
         West Midlands Police             690
dtype: 

In [ ]:
# find the number of duplicates by each crime type
crime_df[
    crime_df.duplicated(keep=False)
]["Crime type"].value_counts()

Crime type
Anti-social behaviour           537472
Violence and sexual offences      3667
Public order                      1825
Drugs                              504
Robbery                            349
Criminal damage and arson          163
Burglary                           147
Vehicle crime                       72
Theft from the person               59
Other theft                         58
Other crime                         35
Shoplifting                         31
Possession of weapons               28
Bicycle theft                        4
Name: count, dtype: int64

In [ ]:
#find the number of crimes with the same ID 
identified_crimes = crime_df[crime_df["Crime ID"].notna()]

print(
    "Records with a Crime ID:",
    len(identified_crimes)
)

print(
    "Duplicate Crime IDs:",
    identified_crimes["Crime ID"].duplicated().sum()
)

Records with a Crime ID: 4257959
Duplicate Crime IDs: 23773


In [ ]:
# viewing the duplicated crime ids 
duplicate_ids = identified_crimes[
    identified_crimes["Crime ID"].duplicated(keep=False)
]

duplicate_ids.sort_values("Crime ID").head(10)

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category
3517152,0008139d6798f3ff9ebd875784cf6aa60483d5be4d06f0...,2025-09,Metropolitan Police Service,Metropolitan Police Service,-0.211128,51.518115,On or near Malton Mews,E01002878,Kensington and Chelsea 003C,Public order,Status update unavailable
3517154,0008139d6798f3ff9ebd875784cf6aa60483d5be4d06f0...,2025-09,Metropolitan Police Service,Metropolitan Police Service,-0.211128,51.518115,On or near Malton Mews,E01002878,Kensington and Chelsea 003C,Public order,Investigation complete; no suspect identified
3511543,0008aa2c3f7f73663f1d5e24a921c485193aab37cdff86...,2025-09,Metropolitan Police Service,Metropolitan Police Service,-0.410480,51.480377,On or near Avenue Close,E01002584,Hounslow 013B,Public order,Unable to prosecute suspect
3511538,0008aa2c3f7f73663f1d5e24a921c485193aab37cdff86...,2025-09,Metropolitan Police Service,Metropolitan Police Service,-0.410480,51.480377,On or near Avenue Close,E01002584,Hounslow 013B,Public order,Status update unavailable
4478886,000b9fef950d66ae052c22dfb299797299e1d81e0bde36...,2026-04,Metropolitan Police Service,Metropolitan Police Service,-0.100434,51.448408,On or near Guernsey Grove,E01003168,Lambeth 025C,Public order,Under investigation
4478888,000b9fef950d66ae052c22dfb299797299e1d81e0bde36...,2026-04,Metropolitan Police Service,Metropolitan Police Service,-0.100434,51.448408,On or near Guernsey Grove,E01003168,Lambeth 025C,Public order,Under investigation
3974238,00106397c0c786dc49bc6bc42627a017e6399398475bf6...,2025-12,Metropolitan Police Service,Metropolitan Police Service,-0.131247,51.514279,On or near Nightclub,E01004763,Westminster 013B,Drugs,Status update unavailable
3974272,00106397c0c786dc49bc6bc42627a017e6399398475bf6...,2025-12,Metropolitan Police Service,Metropolitan Police Service,-0.131247,51.514279,On or near Nightclub,E01004763,Westminster 013B,Drugs,Local resolution
4480341,00144a15948114d352b8ed21ff5df59ffc56b50609e30f...,2026-04,Metropolitan Police Service,Metropolitan Police Service,-0.034179,51.482468,On or near Alexandra,E01003246,Lewisham 002A,Public order,Under investigation
4480335,00144a15948114d352b8ed21ff5df59ffc56b50609e30f...,2026-04,Metropolitan Police Service,Metropolitan Police Service,-0.034179,51.482468,On or near Alexandra,E01003246,Lewisham 002A,Public order,Unable to prosecute suspect


In [ ]:
# duplicated ids but when the outcomes are different
outcomes_per_id = duplicate_ids.groupby("Crime ID")[
    "Last outcome category"
].nunique(dropna=False)

print(
    "Crime IDs with multiple outcomes:",
    (outcomes_per_id > 1).sum()
)

Crime IDs with multiple outcomes: 19038


In [ ]:

columns_to_check = [
    "Month",
    "Reported by",
    "Longitude",
    "Latitude",
    "LSOA code",
    "Crime type"
]

differences_per_id = duplicate_ids.groupby("Crime ID")[columns_to_check].nunique()

print("Crime IDs with different months:",
      (differences_per_id["Month"] > 1).sum())

print("Crime IDs with different police forces:",
      (differences_per_id["Reported by"] > 1).sum())

print("Crime IDs with different crime types:",
      (differences_per_id["Crime type"] > 1).sum())

print("Crime IDs with different LSOA codes:",
      (differences_per_id["LSOA code"] > 1).sum())

Crime IDs with different months: 36
Crime IDs with different police forces: 0
Crime IDs with different crime types: 4634
Crime IDs with different LSOA codes: 2


In [51]:
different_crime_types = differences_per_id[
    differences_per_id["Crime type"] > 1
].index

duplicate_ids[
    duplicate_ids["Crime ID"].isin(different_crime_types)
].sort_values("Crime ID").head(20)

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category
618614,001d74ef9ceb828e814c11416547c3b773eececf091924...,2024-01,Metropolitan Police Service,Metropolitan Police Service,0.022221,51.488552,On or near Supermarket,E01033738,Greenwich 037D,Public order,Investigation complete; no suspect identified
618620,001d74ef9ceb828e814c11416547c3b773eececf091924...,2024-01,Metropolitan Police Service,Metropolitan Police Service,0.022221,51.488552,On or near Supermarket,E01033738,Greenwich 037D,Shoplifting,Status update unavailable
813947,0029dc50f3fa2547c87e395262e9abd51033b7e59da752...,2024-02,Metropolitan Police Service,Metropolitan Police Service,-0.140154,51.516058,On or near Market Place,E01004765,Westminster 013D,Violence and sexual offences,Court result unavailable
813783,0029dc50f3fa2547c87e395262e9abd51033b7e59da752...,2024-02,Metropolitan Police Service,Metropolitan Police Service,-0.140154,51.516058,On or near Market Place,E01004765,Westminster 013D,Shoplifting,Court result unavailable
248208,00479b8385e1d3275d29cbd0f1e35102f8fa623cbdd049...,2023-10,Metropolitan Police Service,Metropolitan Police Service,NaN,NaN,No Location,NaN,NaN,Drugs,Court result unavailable
249567,00479b8385e1d3275d29cbd0f1e35102f8fa623cbdd049...,2023-10,Metropolitan Police Service,Metropolitan Police Service,NaN,NaN,No Location,NaN,NaN,Possession of weapons,Status update unavailable
756233,004fcc20b7c551eb6c8dfb5efad67032376c7eae8abd0d...,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.078582,51.438185,On or near Oakley Drive,E01001618,Greenwich 030C,Criminal damage and arson,Status update unavailable
756241,004fcc20b7c551eb6c8dfb5efad67032376c7eae8abd0d...,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.078582,51.438185,On or near Oakley Drive,E01001618,Greenwich 030C,Violence and sexual offences,Court result unavailable
306950,005d6d5edafdd6525720fffedd6d3ae7d2fcf426b2918d...,2023-11,Metropolitan Police Service,Metropolitan Police Service,0.148744,51.540851,On or near Shopping Area,E01000105,Barking and Dagenham 014C,Violence and sexual offences,Investigation complete; no suspect identified
306933,005d6d5edafdd6525720fffedd6d3ae7d2fcf426b2918d...,2023-11,Metropolitan Police Service,Metropolitan Police Service,0.148744,51.540851,On or near Shopping Area,E01000105,Barking and Dagenham 014C,Robbery,Status update unavailable


In [52]:
# Count how many different months each Crime ID appears in
months_per_id = duplicate_ids.groupby("Crime ID")["Month"].nunique()

print("Crime IDs appearing in multiple months:",
      (months_per_id > 1).sum())

print("Maximum months for one Crime ID:",
      months_per_id.max())

Crime IDs appearing in multiple months: 36
Maximum months for one Crime ID: 2


In [53]:
multiple_month_ids = months_per_id[months_per_id > 1].index

duplicate_ids[
    duplicate_ids["Crime ID"].isin(multiple_month_ids)
].sort_values(["Crime ID", "Month"])[
    ["Crime ID", "Month", "Reported by", "Crime type", "Last outcome category"]
].head(20)

,Crime ID,Month,Reported by,Crime type,Last outcome category
529,121514aa0f4efebb4c107833a463b2a10260913f440617...,2023-09,Lincolnshire Police,Vehicle crime,Investigation complete; no suspect identified
301239,121514aa0f4efebb4c107833a463b2a10260913f440617...,2023-11,Lincolnshire Police,Vehicle crime,Investigation complete; no suspect identified
861669,264ffc70ee27992a166cf5631500061dcd87d3817531fc...,2024-03,Lincolnshire Police,Violence and sexual offences,Status update unavailable
1141480,264ffc70ee27992a166cf5631500061dcd87d3817531fc...,2024-05,Lincolnshire Police,Violence and sexual offences,Unable to prosecute suspect
448176,2a8380d6ebf0028fefd3c10a2b750bbbdae39fe2c15a3d...,2023-12,Lincolnshire Police,Criminal damage and arson,Unable to prosecute suspect
1142465,2a8380d6ebf0028fefd3c10a2b750bbbdae39fe2c15a3d...,2024-05,Lincolnshire Police,Criminal damage and arson,Unable to prosecute suspect
4274,2e97b0c3d1ec3e32d09452ce94a27b8cea349cf179d0c2...,2023-09,Lincolnshire Police,Vehicle crime,Investigation complete; no suspect identified
304723,2e97b0c3d1ec3e32d09452ce94a27b8cea349cf179d0c2...,2023-11,Lincolnshire Police,Vehicle crime,Investigation complete; no suspect identified
725259,2ecfe71ebd8a7bc9a64c3223e8b8f3acd6a854e6f4b7e9...,2024-02,Lincolnshire Police,Shoplifting,Status update unavailable
1140899,2ecfe71ebd8a7bc9a64c3223e8b8f3acd6a854e6f4b7e9...,2024-05,Lincolnshire Police,Violence and sexual offences,Local resolution


In [54]:
print("Rows before removing exact duplicates:", len(crime_df))

print("Exact duplicate occurrences:",
      crime_df.duplicated().sum())

Rows before removing exact duplicates: 5150729
Exact duplicate occurrences: 370933


In [55]:
crime_df_deduplicated = crime_df.drop_duplicates()

print("Rows after removing exact duplicates:",
      len(crime_df_deduplicated))

print("Rows removed:",
      len(crime_df) - len(crime_df_deduplicated))

Rows after removing exact duplicates: 4779796
Rows removed: 370933


In [56]:
exact_duplicates = crime_df[
    crime_df.duplicated(keep=False)
]

print(
    "Duplicate-flagged rows with missing Crime ID:",
    exact_duplicates["Crime ID"].isna().sum()
)

print(
    "Duplicate-flagged rows with a Crime ID:",
    exact_duplicates["Crime ID"].notna().sum()
)

Duplicate-flagged rows with missing Crime ID: 537472
Duplicate-flagged rows with a Crime ID: 6942


In [57]:
crime_df = crime_df.drop_duplicates()

print("Rows after removing exact duplicates:", len(crime_df))
print("Remaining exact duplicates:", crime_df.duplicated().sum())

Rows after removing exact duplicates: 4779796
Remaining exact duplicates: 0


In [58]:
missing_values = crime_df.isna().sum()

print(missing_values[missing_values > 0])

Crime ID                 525678
Longitude                 35711
Latitude                  35711
LSOA code                 35712
LSOA name                 35712
Last outcome category    525678
dtype: int64


In [59]:
missing_location = crime_df[
    crime_df["Longitude"].isna()
    | crime_df["Latitude"].isna()
]

print("Records with missing coordinates:", len(missing_location))

print("Also missing LSOA code:",
      missing_location["LSOA code"].isna().sum())

print("Also missing LSOA name:",
      missing_location["LSOA name"].isna().sum())

Records with missing coordinates: 35711
Also missing LSOA code: 35711
Also missing LSOA name: 35711


In [60]:
missing_location["Location"].value_counts(dropna=False).head(10)

Location
No Location    35711
Name: count, dtype: int64

In [61]:
crime_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4779796 entries, 0 to 5150728
Data columns (total 11 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Crime ID               object 
 1   Month                  object 
 2   Reported by            object 
 3   Falls within           object 
 4   Longitude              float64
 5   Latitude               float64
 6   Location               object 
 7   LSOA code              object 
 8   LSOA name              object 
 9   Crime type             object 
 10  Last outcome category  object 
dtypes: float64(2), object(9)
memory usage: 437.6+ MB


In [62]:
crime_df["Month"] = pd.to_datetime(crime_df["Month"], format="%Y-%m")

In [63]:
crime_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4779796 entries, 0 to 5150728
Data columns (total 11 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   Crime ID               object        
 1   Month                  datetime64[ns]
 2   Reported by            object        
 3   Falls within           object        
 4   Longitude              float64       
 5   Latitude               float64       
 6   Location               object        
 7   LSOA code              object        
 8   LSOA name              object        
 9   Crime type             object        
 10  Last outcome category  object        
dtypes: datetime64[ns](1), float64(2), object(8)
memory usage: 437.6+ MB


In [64]:
text_columns = [
    "Reported by",
    "Falls within",
    "Location",
    "LSOA code",
    "LSOA name",
    "Crime type",
    "Last outcome category"
]

for column in text_columns:
    print(f"\n{column}:")
    print("Unique values:", crime_df[column].nunique(dropna=False))
    print("Missing values:", crime_df[column].isna().sum())


Reported by:
Unique values: 4
Missing values: 0

Falls within:
Unique values: 4
Missing values: 0

Location:
Unique values: 70800
Missing values: 0

LSOA code:
Unique values: 13892
Missing values: 35712

LSOA name:
Unique values: 13892
Missing values: 35712

Crime type:
Unique values: 14
Missing values: 0

Last outcome category:
Unique values: 16
Missing values: 525678


In [65]:
print("Police forces:")
print(sorted(crime_df["Reported by"].unique()))

print("\nCrime types:")
print(sorted(crime_df["Crime type"].unique()))

Police forces:
['Lincolnshire Police', 'Metropolitan Police Service', 'South Yorkshire Police', 'West Midlands Police']

Crime types:
['Anti-social behaviour', 'Bicycle theft', 'Burglary', 'Criminal damage and arson', 'Drugs', 'Other crime', 'Other theft', 'Possession of weapons', 'Public order', 'Robbery', 'Shoplifting', 'Theft from the person', 'Vehicle crime', 'Violence and sexual offences']
